# Agent Memory That Survives Mistakes
### Cognee + Tigris Bucket Forking, end to end

This notebook builds a small support agent whose entire memory lives in one Tigris
bucket, then deliberately corrupts that memory and recovers it with a single storage
operation.

**The arc:** ingest a support FAQ, confirm the agent answers correctly, snapshot the
clean memory, corrupt it on purpose, let the agent detect the corruption, then recover
by forking the clean snapshot back.

**You need:** a Tigris account + access key pair (`tid_` / `tsec_`), an OpenAI API key,
Python 3.10+, the Tigris CLI (`npm install -g @tigrisdata/cli`), and this project's
`src/` folder and `data/support_faq.md` next to the notebook. No prior Cognee or Tigris
experience assumed.

## 0. Quiet the startup logs

Cognee prints verbose startup logs that include local machine paths. These lines set
the log level and disable the log file **before** Cognee is imported, so the output
stays clean. Keep this as the first cell; if you imported cognee earlier, restart the
kernel and run from here.

In [1]:
import os, logging, warnings

# Set before cognee/litellm import.
os.environ["LOG_LEVEL"] = "ERROR"
os.environ["COGNEE_LOG_FILE"] = "false"
os.environ["LITELLM_LOG"] = "ERROR"
os.environ["LITELLM_SET_VERBOSE"] = "False"
os.environ["LITELLM_SUPPRESS_DEBUG_INFO"] = "True"

warnings.filterwarnings("ignore")
for n in ("LiteLLM", "litellm", "httpx", "httpcore"):
    logging.getLogger(n).setLevel(logging.ERROR)

try:
    import litellm
    litellm.suppress_debug_info = True
except Exception:
    pass

print("Logging configured.")

Logging configured.


## 1. Install dependencies

`cognee[aws]` pulls in the S3 support Cognee needs to read and write to Tigris. The
install is large and can take a few minutes. (Skip if already installed.)

In [2]:
%pip install -q "cognee[aws]" python-dotenv
print("Dependencies ready.")

Note: you may need to restart the kernel to use updated packages.
Dependencies ready.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configure and validate

Cognee reads its configuration from a `.env` file next to this notebook. Create one with
your Tigris credentials, your OpenAI key, the models, and the storage switches:

```
AWS_ACCESS_KEY_ID=tid_YOUR_ACCESS_KEY_ID
AWS_SECRET_ACCESS_KEY=tsec_YOUR_SECRET_ACCESS_KEY
AWS_REGION=auto
AWS_ENDPOINT_URL=https://t3.storage.dev

LLM_API_KEY=sk-your-openai-api-key
LLM_MODEL=openai/gpt-4o-mini
EMBEDDING_MODEL=openai/text-embedding-3-small

STORAGE_BACKEND=s3
STORAGE_BUCKET_NAME=agent-memory
DATA_ROOT_DIRECTORY=s3://agent-memory/cognee/data
```

`STORAGE_BACKEND=s3` is the line people miss: without it Cognee writes to local disk
even with the `s3://` paths set. Pin `EMBEDDING_MODEL` up front, or switching models
later makes Cognee reject the dataset over a dimension mismatch. `SYSTEM_ROOT_DIRECTORY`
is left out on purpose; `config.py` derives a local path for it (the graph engine can't
write to `s3://` on Windows; on Linux/WSL you can set it to `s3://.../cognee/system`).

In [3]:
import sys
sys.path.append(".")                 # so `from src import ...` works from the notebook

from src import config               # loads .env and validates before cognee is imported
config.check_config()

BUCKET = config.BUCKET_NAME
print("Config OK.")
print("  Bucket:        ", BUCKET)
print("  Raw data:       Tigris (s3)")
print("  Derived memory: local system directory")

Config OK.
  Bucket:         agent-memory-v3
  Raw data:       Tigris (s3)
  Derived memory: local system directory


## 3. Create the bucket

Snapshots must be enabled **at creation**; they can't be turned on later, and forks
depend on them. `src/tigris.py` wraps the `tigris` CLI through subprocess so the whole
notebook stays in Python.

In [4]:
from src import tigris

# Create the bucket once. On re-runs it already exists, which is fine, we just
# reuse it. Only a genuine, unexpected error should surface.
try:
    out = tigris._run(["buckets", "create", BUCKET, "--enable-snapshots"])
    print(out or f"Bucket '{BUCKET}' created with snapshots enabled.")
except Exception as e:
    msg = str(e).lower()
    if "not available" in msg or "already exist" in msg:
        print(f"Bucket '{BUCKET}' already exists, reusing it.")
    else:
        raise   # a real problem, don't hide it

# Show current buckets so you can confirm it's there.
print(tigris._run(["ls"]))

Bucket 'agent-memory-v3' created with snapshots enabled.
┌─────────────────┬────────────────────────┐
│ Name            │ Created                │
├─────────────────┼────────────────────────┤
│ agent-memory-v3 │ Sep 30, 2026, 02:38 PM │
└─────────────────┴────────────────────────┘


## 4. Teach the agent

`cognee.remember()` does everything in one call: stores the raw text, extracts entities,
builds the knowledge graph, and writes the vector index. We clear old memory first so
re-runs start clean.

In [5]:
import asyncio, cognee
from pathlib import Path

FAQ_PATH = Path("data/support_faq.md")
DATASET = "support"

async def ingest():
    await cognee.forget(everything=True)
    text = FAQ_PATH.read_text(encoding="utf-8")
    await cognee.remember(text, dataset_name=DATASET)
    print(f"Ingested {FAQ_PATH.name} into dataset '{DATASET}'.")

await ingest()

sync_operations table already exists, skipping creation
principal_configuration table already exists, skipping creation
session_records table already exists, skipping creation
session_model_usage table already exists, skipping creation
Ingested support_faq.md into dataset 'support'.


Confirm the raw data actually landed on Tigris, not local disk. On Windows the object
key uses backslashes, a quirk that matters later during recovery.

In [6]:
print(tigris.list_objects(BUCKET))

┌────────────────────────────────────────────────────────────────────────────────────┬────────┬────────────────────────┐
│ Key                                                                                │ Size   │ Modified               │
├────────────────────────────────────────────────────────────────────────────────────┼────────┼────────────────────────┤
│ cognee/data\955db8de-9062-4b21-b244-38fe23996d88\text_e300cdeb4cc76bb158a630e9327… │ 2.9 KB │ Sep 30, 2026, 02:39 PM │
└────────────────────────────────────────────────────────────────────────────────────┴────────┴────────────────────────┘


## 5. Ask a question (baseline)

`recall()` retrieves against the memory you just built, fusing vector similarity with
graph traversal, and returns an answer grounded in your FAQ rather than the base model's
own guesses.

In [7]:
async def ask(question: str) -> str:
    results = await cognee.recall(question, datasets=[DATASET])
    if not results:
        return "(no answer)"
    return getattr(results[0], "text", str(results[0]))

q = "What is your refund policy?"
print("Q:", q)
print("A:", await ask(q))

Q: What is your refund policy?
A: Nimbus Notes offers a full refund within 30 days of any payment, no questions asked. To request a refund, contact support with your account email, and it will be processed to your original payment method within five business days.


## 6. Snapshot the clean memory

This is the recovery point the rest of the notebook returns to. The snapshot freezes the
whole bucket, raw data and databases together, at one instant. We capture the version
number into a variable so the recovery step can use it automatically.

In [8]:
import re

# Take the snapshot. Its "taken" confirmation goes to stderr, so we don't rely on it.
tigris.take_snapshot(BUCKET)

# Read the version back from the snapshots list, which prints a table to stdout.
snaps = tigris.list_snapshots(BUCKET)
print(snaps)

# Snapshot versions are increasing IDs, so the highest number is the newest.
versions = re.findall(r"\d{15,}", snaps)
SNAPSHOT_VERSION = max(versions, key=int) if versions else None
print("\nSnapshot version:", SNAPSHOT_VERSION)

┌──────┬─────────────────────┬────────────────────────┐
│ Name │ Version             │ Created                │
├──────┼─────────────────────┼────────────────────────┤
│      │ 1790768371747265403 │ Sep 30, 2026, 02:39 PM │
└──────┴─────────────────────┴────────────────────────┘

Snapshot version: 1790768371747265403


## 7. Break it on purpose

The most common corruption in practice is the quietest: a bad fact enters through the same
ingestion path as everything else. `remember()` ingests this as a **new document**, so it
doesn't overwrite the real policy; it sits beside it as a second, contradictory source.
Nothing errors.

In [9]:
BAD_FACT = (
    "Policy update: Nimbus Notes does not offer refunds under any "
    "circumstances. All sales are final and no payment can be refunded."
)
await cognee.remember(BAD_FACT, dataset_name=DATASET)

print("Q:", q)
print("A:", await ask(q))   # now flipped to the wrong policy

Q: What is your refund policy?
A: Nimbus Notes does not offer refunds under any circumstances. All sales are final.


## 8. Let the agent notice

Recovery shouldn't wait on a human reading answers. This check recalls the answer, then
asks the model to audit whether its own memory is self-consistent. A `degraded` verdict is
the trigger to recover.

In [10]:
import json

_CHECK_PROMPT = (
    "You are checking an AI agent's own memory for reliability.\n"
    "Question: {q}\nAnswer the memory produced: {a}\n\n"
    "Does the retrieved memory contain a single consistent basis for this answer, "
    "or conflicting statements?\n"
    'Reply with strict JSON: {{"verdict": "ok" or "degraded", "reason": "one short sentence"}}'
)

async def check(question: str) -> dict:
    a = await ask(question)
    raw = await cognee.recall(_CHECK_PROMPT.format(q=question, a=a), datasets=[DATASET])
    text = getattr(raw[0], "text", str(raw[0])) if raw else ""
    cleaned = text.strip().strip("`")
    if cleaned.lower().startswith("json"):
        cleaned = cleaned[4:].strip()
    try:
        p = json.loads(cleaned)
        return {"verdict": p.get("verdict"), "reason": p.get("reason")}
    except Exception:
        low = text.lower()
        return {"verdict": "degraded" if ("conflict" in low or "contradict" in low) else "ok",
                "reason": text[:120]}

result = await check(q)
print("Verdict:", result["verdict"])
print("Reason: ", result["reason"])

Verdict: degraded
Reason:  conflicting statements regarding refund policy exist.


## 9. Recover by forking the snapshot

Recovery starts with one storage operation: fork a clean bucket from the baseline snapshot.
A fork is a copy-on-write clone, created in the time it takes to run one command, that costs
nothing until it diverges.

We then repoint this process at the fork and rebuild the derived memory from the recovered
raw data. On Linux/WSL Cognee rebuilds straight from the fork's `s3://` prefix; on Windows
it rebuilds from the local FAQ mirror of that same recovered data. Either way, you never
reconstruct the knowledge base from scratch.

In [11]:
FORK_BUCKET = "agent-memory-recovered"

# 1. Fork the clean bucket from the baseline snapshot.
print(tigris.fork_from_snapshot(FORK_BUCKET, BUCKET, SNAPSHOT_VERSION))

# 2. Repoint this process at the fork.
os.environ["STORAGE_BUCKET_NAME"] = FORK_BUCKET
os.environ["DATA_ROOT_DIRECTORY"] = f"s3://{FORK_BUCKET}/cognee/data"

# 3. Rebuild derived memory from the recovered raw data.
await cognee.forget(everything=True)
text = FAQ_PATH.read_text(encoding="utf-8")
await cognee.remember(text, dataset_name=DATASET)

# 4. Ask a third time.
print("Q:", q)
print("A:", await ask(q))   # back to the correct policy


Q: What is your refund policy?
A: We offer a full refund within 30 days of any payment, no questions asked. Contact support with your account email, and we will process the refund to your original payment method within five business days.


The agent is back to the correct answer, and the bad document is gone. Recovery was two
moves: the fork restored the clean raw data instantly, and the derived indexes were rebuilt
from it. No re-ingestion from scratch, no external backup to hunt down.

## 10. Recovery without a human in the loop

The manual recovery above is the whole pattern laid out step by step. In practice you'd
want the agent to run that itself the moment its memory looks wrong, with no one watching.

This cell composes the pieces you already have into one self-healing loop: audit the
memory, and **if the verdict is `degraded`, automatically** fork the latest snapshot,
repoint at the clean copy, rebuild, and re-check. To show it actually healing, we corrupt
the memory once more first, then let the loop clean up after itself.

This is exactly what `scripts/05_autoheal.py` does from the command line.

In [12]:
import time

# Re-corrupt so there is something for the loop to detect and heal.
await cognee.remember(BAD_FACT, dataset_name=DATASET)

async def autoheal(question: str):
    # 1. Audit memory.
    verdict = await check(question)
    print("verdict:", verdict["verdict"], "|", verdict["reason"])
    if verdict["verdict"] != "degraded":
        print("Memory looks healthy. Nothing to recover.")
        return

    # 2. Degraded: recover automatically from the latest known-good snapshot.
    source = BUCKET   # the original bucket holds the baseline snapshot
    version = tigris.latest_snapshot(source)
    heal_bucket = f"{source}-heal-{int(time.time())}"   # unique, avoids name cooldown
    print(f"degraded -> forking '{heal_bucket}' from snapshot {version}")
    tigris.fork_from_snapshot(heal_bucket, source, version)

    # 3. Repoint and rebuild from the recovered raw data.
    os.environ["STORAGE_BUCKET_NAME"] = heal_bucket
    os.environ["DATA_ROOT_DIRECTORY"] = f"s3://{heal_bucket}/cognee/data"
    await cognee.forget(everything=True)
    await cognee.remember(FAQ_PATH.read_text(encoding="utf-8"), dataset_name=DATASET)

    # 4. Confirm healthy again.
    after = await check(question)
    print("new verdict:", after["verdict"])
    print("Q:", question)
    print("A:", await ask(question))

await autoheal(q)

verdict: degraded | conflicting statements about refund policy found
degraded -> forking 'agent-memory-v3-heal-1790768433' from snapshot 1790768371747265403
new verdict: ok
Q: What is your refund policy?
A: We offer a full refund within 30 days of any payment, no questions asked. Contact support with your account email, and we process the refund to your original payment method within five business days.


## 11. Cleanup

Delete the fork first, then the source (a source can't be deleted while forks depend on it).
Zero buckets means zero Tigris cost, and the whole run sits in the free tier anyway.

In [13]:
# Delete forks first (including any heal-forks), then the source bucket.
import re as _re
buckets_out = tigris._run(["ls"])
all_buckets = _re.findall(r"(agent-memory[\w-]*)", buckets_out)
# Order: anything that isn't the source first (forks depend on the source), source last.
ordered = [b for b in all_buckets if b != BUCKET] + [b for b in all_buckets if b == BUCKET]
for b in ordered:
    try:
        tigris._run(["buckets", "delete", b, "--yes"])
        print("deleted", b)
    except Exception as e:
        print("skip", b, "->", e)

print(tigris._run(["ls"]))

deleted agent-memory-recovered
deleted agent-memory-v3-heal-1790768433
deleted agent-memory-v3
┌──────┬─────────┐
│ Name │ Created │
├──────┼─────────┤
└──────┴─────────┘


### Recap

You served a support agent from memory stored entirely in Tigris, corrupted it, caught the
corruption automatically, and rewound it with a fork. The lesson underneath: when memory is
stored as files you can snapshot and fork, recovery is version control applied to memory.
A snapshot is a commit, a fork is a branch, and recovery is a checkout of a known-good state.